# Init

In [2]:
# imports

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import cvxpy as cp
import matplotlib.pyplot as plt 
import matplotlib.dates as mdates

c:\Users\phili\miniconda3\envs\phpo\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
main_df = pd.read_feather("./Data/US_datastream/US_data_panel_filtered_0.15.feather", columns=["Date", "DSCD", "Return", "MarketCAP"]) # my PC struggeld with reading the whole dataset with all columns, don't know why
main_df = main_df.sort_values(by="Date")

# Helpers

In [4]:
# metrics for stock_selection 

def metric_sharpe(panel: pd.DataFrame):
    ret_matrix = construct_ret_matrix(panel=panel)
    return ret_matrix.mean() / ret_matrix.std(ddof=1) 

def metric_mcap(panel: pd.DataFrame):
    return (
        panel.sort_values("Date").groupby("DSCD")["MarketCAP"].last()
    )


In [5]:
# helpful functions

# function for changing the format of the initial DataFrame

def construct_ret_matrix(panel: pd.DataFrame) -> pd.DataFrame:

    return panel.pivot(index="Date", columns="DSCD", values="Return")

# filter for stocks with available return data

def filter_av_returns(panel: pd.DataFrame) -> list:

    total_days = panel["Date"].nunique()

    valid_stocks = []
    for dscd, dscd_panel in panel.groupby("DSCD", sort=False):
        
        dscd_filtered_series = dscd_panel["Return"].dropna()
        #dscd_filtered_series = dscd_filtered_series[dscd_filtered_series != 0]

        if len(dscd_filtered_series) != total_days: # I don't allow even a single missing return (but 0 returns are in for now) -> smaler scope helps me here
            continue
        
        valid_stocks.append(dscd)
    
    return valid_stocks

# my individual stock selection function, should also work variable by giving it different metrics

def select_stocks(panel: pd.DataFrame, nstocks: int, metric: callable, by_highest: bool) -> list:
    
    scores = metric(panel)
    return scores.sort_values(ascending=(not by_highest)).head(nstocks).index.tolist()

In [21]:
def sample_cov(R: pd.DataFrame) -> pd.DataFrame:
    
    Sigma = R.cov().values
    Sigma = Sigma + 1e-6 * np.eye(Sigma.shape[0]) 
    return pd.DataFrame(Sigma, index=R.columns, columns=R.columns)

`sample_cov()`

for stocks i,j compute:

$$\Sigma_{i,j} = \frac{1}{T-1} \sum_t{(r_{i,t} - \bar{r}_i)(r_{j,t} - \bar{r}_j)}$$

In [7]:
def sample_mu(R: pd.DataFrame) -> pd.Series:

    return R.mean()

`sample_mu`

for stock i compute:

$$\bar{r}_i = \frac{1}{T} \sum_t{r_{i,t}}$$

In [17]:
def min_var_weights(mu: pd.Series, Sigma: pd.DataFrame) -> pd.Series:
    
    S = cp.psd_wrap(Sigma.values) # use ndarray instead of DF and trust to be positive semidefinite
    
    X = cp.Variable(S.shape[0]) # x_i for each stock i in S

    constraints = [X >= 0,
                   X <= 1,
                   X.sum() == 1.0]
    
    opti_problem = cp.Problem(cp.Minimize(cp.quad_form(X,S)), constraints=constraints)

    opti_problem.solve()

    # Error handling
    if opti_problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
        print(f"min_var optimisation skipped. Status: {opti_problem.status}")
        return None
    
    return pd.Series(X.value.round(6), index=Sigma.index)

`min_var_weights()`

solve:
$$\min_x{x^T\sum{x}}$$
w.r.t.:
$$\text{full investment}$$
$$\sum_i{x_i}=1$$
$$\text{long-only}$$
$$0 \leq x_i \leq 1$$

In [16]:
# TODO check for correctness and why it's correct

def max_sharpe_weights(mu: pd.Series, Sigma: pd.DataFrame) -> pd.Series:
    
    # Alignment
    stocks = mu.index
    Sigma = Sigma.loc[stocks,stocks]

    # Excess Return (ignored for now)
    risk_free_rate = 0.0
    mu_excess = mu - risk_free_rate

    nstocks = len(stocks)

    S = cp.psd_wrap(Sigma.values) # again the positive semidefinite thing. Ridge has been added to diagonal in cov estimation

    """
    Situation: we search for w so that Sharpe is maximized. We therefore need to maximize a fraction -> problematic because not linear
    Trick: minimize risk with vars y,k so that y = k * w, since then w = y / k
    """

    y = cp.Variable(nstocks) # our "reformed" weights
    k = cp.Variable(nonneg=True)

    constraints = [
        mu_excess.values @ y == 1, # key for our trick, nominator = 1 allows us to just minimize riks (denominator) to max the fraction
        cp.sum(y) == k, # in order to do w = y / k later, sum(y / k) has to be 1 which is sum(y) = k
        y >= 0 * k, # long-only, has to be relative to k (does not matter here since my testing lowerbound is 0)
        y <= 1 * k, # no overinvesting, also relative to k 
        k >= 1e-8 # since we divide by k later k has to be > 0 
    ]

    prob = cp.Problem(cp.Minimize(cp.quad_form(y, S)), constraints=constraints)
    prob.solve()

    if prob.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
        print(f"Optimisation skipped: {prob.status}")
        return None
    
    w = y.value / k.value

    return pd.Series(w.round(6), index=Sigma.index)

`max_sharpe_weights()`

solve:
$$\max_w{\frac{(\mu - r_f1)^T w}{\sqrt{w^T\Sigma w}}}$$
hard to find otpimum, therefore:
$$y = kw \Leftrightarrow w = \frac{y}{k}$$
with:
$$\mu_{e}^T y = 1$$
now solve just denominator:
$$\min_{y,k}{y^T\Sigma y}$$
w.r.t.:
$$\mu_{e}^T y = 1$$
$$1^T y = k$$
$$0 \leq y \leq k\forall y$$
$$k > 0$$

In [10]:
def calc_pf_returns(R: pd.DataFrame, w: pd.Series):

    return R.loc[:, w.index].values @ w.values # cool syntax

`calc_pf_returns`

multiply the returns of each stock for every with the corresponding daily return:

$$Rw$$

$$\text{e.g.}$$

$$R=\begin{bmatrix} 0,01 & 0,02 & -0,01 \\ -0,03 & 0,01 & 0,04 \end{bmatrix}$$

$$w=\begin{bmatrix} 0,5 \\ 0,3 \\ 0,2 \end{bmatrix}$$

$$\text{which gives}$$

$$r_{t_1}=0,01*0,5+0,02*0,3+(-0,01)*0,2=0,009$$

$$r_{t_2}=(-0,03)*0,5+0,01*0,3+0,04*0,2=-0,004$$

# Simulation

In [ ]:
# everything regarding the simulated investors

# my individual investor class

class Investor:

    def __init__(self, strategy: str, cov_estimator: str, metric:str):
        
        # attributes

        self.strategy = strategy
        self.cov_estimator = cov_estimator
        self.weight_history = []
        self.return_history = []
        self.current_weights = None

        slippage = 0.001
        fee = 0.0005
        self.total_cost = slippage + fee # for now I'm implementing proportional TK with a fixed cost rate, however TK not yet included in optimization

        if self.strategy == "maxsharp":
            self.weights = max_sharpe_weights
        else: # default strategy = min_var
            self.weights = min_var_weights
            
        if self.cov_estimator == "TODO":
            pass
        else: # default covariance estimation is in-sample 
            self.covar = sample_cov

        if metric == "mcap":
            self.metric = metric_mcap
        else: # default selection metric is pseudo Sharpe
            self.metric = metric_sharpe

    def add_pf_returns(self, ret_mat: pd.DataFrame, weights: pd.Series, turnover):
        
        R = ret_mat
        w = weights

        pf_returns = calc_pf_returns(R=R, w=w) 

        cost = self.total_cost * turnover
        pf_returns[0] = pf_returns[0] - cost

        self.return_history.append(pd.Series(pf_returns, index=R.index))


In [ ]:
# main programm

df = main_df[["Date", "Return","DSCD", "MarketCAP"]]

un_dates = np.sort(df["Date"].unique())
T = len(un_dates)
un_dates_cut = un_dates # just to cut down a bit for this experimental scope
T_cut = len(un_dates_cut)

estimation_window = 512
holding_period = 21

times_to_rebalance = range(estimation_window, T_cut, holding_period)

Investors = {
    "Max_Sharpe_selS": Investor(strategy="maxsharp", cov_estimator="Sample", metric="pSharpe"), # selS for Sharpe Selection
    "Min_Var_selS": Investor(strategy="minvar", cov_estimator="Sample", metric="pSharpe"),
    "Max_Sharpe_selM": Investor(strategy="maxsharp", cov_estimator="Sample", metric="mcap"), # selM for MCAP Selection
    "Min_Var_selM": Investor(strategy="minvar", cov_estimator="Sample", metric="mcap")
}

Metrics = {
    "pSharpe": metric_sharpe,
    "mcap": metric_mcap
}

for t in tqdm(times_to_rebalance):

    est_start = un_dates_cut[t - estimation_window]
    est_end = un_dates_cut[t - 1]

    prevent_overflow_idx = min(t + holding_period, T_cut)

    holding_dates = un_dates_cut[t:prevent_overflow_idx]

    estimation_sector = df[(df["Date"] >= est_start) & (df["Date"] <= est_end)]

    filtered_stocks = filter_av_returns(estimation_sector)
    filtered_sector = estimation_sector[estimation_sector["DSCD"].isin(filtered_stocks)]
    filtered_sector_R = construct_ret_matrix(panel=filtered_sector)
    
    for strat, inv in Investors.items():

        selected_stocks = select_stocks(panel=filtered_sector, nstocks=10, metric=inv.metric, by_highest=True)
        selected_sector = filtered_sector[filtered_sector["DSCD"].isin(selected_stocks)]
        selected_sector_R = filtered_sector_R.loc[:, selected_stocks] # should be the same as just caling construct_ret_matrix again
        selected_sector_R = selected_sector_R.loc[:, selected_sector_R.columns.sort_values()] # not sure why we do this exavtly, I think because rows should match for our matrix calculations later -> added it to open questions

        # build Portfolio

        sigma = inv.covar(R=selected_sector_R)

        weights = inv.weights(mu=sample_mu(selected_sector_R), Sigma=sigma)

        if weights is None:
            inv.weight_history.append(None)
            inv.return_history.append(pd.Series(index=selected_sector_R.index, dtype=float))

        sorted_dscds = weights.index.tolist()

        # TK

        if inv.current_weights is None:
            turnover = np.abs(weights).sum()
        else:
            old, new = inv.current_weights.align(weights, fill_value=0)
            turnover = np.abs(new - old).sum()
        
        inv.current_weights = weights

        # hold

        panel_held = df[(df["Date"].isin(holding_dates)) & (df["DSCD"].isin(selected_stocks))]
        
        R_held = construct_ret_matrix(panel=panel_held).reindex(index=holding_dates, columns=sorted_dscds).fillna(0) # fillna(0) problematic

        inv.weight_history.append({
            "Date": holding_dates[0],
            "w": weights.copy()
        })

        inv.add_pf_returns(ret_mat=R_held, weights=weights, turnover=turnover)

    #break # for testing

# Evaluation

In [19]:
# Summary and Visualization 

def evaluate_pf_investor(investor: Investor):

    pf_returns_series = pd.concat(investor.return_history)
    annual_factor = 252 

    mean = pf_returns_series.mean() * annual_factor
    std = pf_returns_series.std(ddof=1) * np.sqrt(annual_factor)

    risk_free_rate = 0.00 / annual_factor # 2% risk free rate
    excess_returns = pf_returns_series - risk_free_rate

    sharpe = (excess_returns.mean() / excess_returns.std(ddof=1)) * np.sqrt(annual_factor)

    pf_evaluation = pd.Series({
        "Annual Mean":  mean,
        "Annual Std":   std,
        "Annual Sharpe": sharpe
    })

    return pf_evaluation

def plot_pf_investors_wealth():

    figsize = (10, 10)

    fig, ax = plt.subplots(figsize=figsize)

    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"] # get colors from mpl, unnecassary here since I'm starting with plotting just one pf

    for i, (strat, inv) in enumerate(Investors.items()):

        pf_returns_series = pd.concat(inv.return_history)
        wealth_series = (1 + pf_returns_series).cumprod() # important

        ax.plot(
            wealth_series.index,
            wealth_series.values,
            label=strat,
            color=colors[i],
            linewidth=1.3
    )

    ax.set_title("Cumulative Wealth")
    ax.set_ylabel("Growth of 1$")
    ax.axhline(1.0, color="black", linewidth=0.6, linestyle="--")
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

    fig.tight_layout()
    plt.show()

def w_to_matrix(investor:Investor):

    return pd.DataFrame({elem["Date"]: elem["w"] for elem in investor.weight_history})


In [ ]:
plot_pf_investors_wealth()
